# Notebook 00 — Phát biểu bài toán

Notebook này khảo sát dữ liệu và làm rõ yêu cầu của bài toán.
Không huấn luyện mô hình, không đo đạc kết quả.

Nội dung trình bày:
- Dữ liệu gồm những thành phần nào
- Hệ thống cần thực hiện nhiệm vụ gì
- Kết quả được chấm điểm ra sao
- Mốc điểm nào được coi là đạt yêu cầu

**Cuộc thi:** RETECO — SemEval 2027, Track 1a (Temporal Grounded Retrieval).

## Tóm tắt bài toán

Bài toán thuộc nhóm **truy hồi thông tin** (information retrieval), tương tự
cơ chế của một công cụ tìm kiếm nhưng trên một kho văn bản giới hạn.

- **Đầu vào:** một câu hỏi.
- **Đầu ra:** danh sách văn bản được xếp hạng theo mức độ liên quan.
- **Yêu cầu:** văn bản đúng phải nằm ở những vị trí đầu tiên.

Điểm đặc thù của cuộc thi: câu hỏi chứa **ràng buộc về thời gian**.
Ví dụ: *"tính đến năm 2017"*, *"sau lần cập nhật gần nhất"*.

Một văn bản đúng chủ đề nhưng sai mốc thời gian vẫn bị tính là **không liên quan**.
Đây là phần khó nhất của bài toán.

## 1. Kết nối tới dữ liệu

In [1]:
from pathlib import Path
import json
import re
from math import log2

# Change this path if the dataset is stored elsewhere
DATA = Path(r"D:\RETECO-project\reteco_data\track1_tempo")

if DATA.is_dir():
    print("Dataset found:", DATA)
else:
    print("DATASET NOT FOUND at:", DATA)
    print("Please update the DATA variable above.")

Dataset found: D:\RETECO-project\reteco_data\track1_tempo


## 2. Cấu trúc theo nhóm chủ đề

Dữ liệu được chia thành 13 nhóm chủ đề độc lập (domain), thuộc bốn lĩnh vực:

| Lĩnh vực | Các nhóm |
| --- | --- |
| Blockchain | bitcoin, cardano, iota, monero |
| Khoa học xã hội | economics, law, politics, history |
| Ứng dụng | quant, travel, workplace, genealogy |
| Khoa học kỹ thuật | hsm (lịch sử khoa học và toán học) |

Việc truy hồi diễn ra **độc lập trong từng nhóm**. Câu hỏi thuộc nhóm nào thì
chỉ tìm trong kho văn bản của nhóm đó.

In [2]:
domains = sorted(p.name for p in DATA.iterdir() if p.is_dir())

print(f"Number of domains: {len(domains)}")
for name in domains:
    print("  -", name)

Number of domains: 13
  - bitcoin
  - cardano
  - economics
  - genealogy
  - history
  - hsm
  - iota
  - law
  - monero
  - politics
  - quant
  - travel
  - workplace


## 3. Các file trong một nhóm

In [3]:
sample_domain = DATA / domains[0]

print(f"Files in domain '{domains[0]}':\n")
for f in sorted(sample_domain.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:<28} {size_mb:>8.1f} MB")

Files in domain 'bitcoin':

  documents.jsonl                 403.5 MB
  examples_dev.jsonl                0.1 MB
  examples_train.jsonl              0.2 MB
  guidance_dev.jsonl                0.1 MB
  guidance_train.jsonl              0.2 MB
  qrels_dev.txt                     0.0 MB
  qrels_steps_dev.txt               0.0 MB
  qrels_steps_train.txt             0.0 MB
  qrels_train.txt                   0.0 MB
  steps_dev.jsonl                   0.0 MB
  steps_train.jsonl                 0.1 MB


Vai trò của từng file:

| File | Nội dung | Track 1a có dùng? |
| --- | --- | --- |
| `documents.jsonl` | Kho văn bản để truy hồi | **Có** |
| `examples_train.jsonl` | Câu hỏi kèm đáp án, dùng để phát triển | **Có** |
| `examples_dev.jsonl` | Câu hỏi dùng để kiểm tra cuối | **Có** |
| `qrels_train.txt` | Đáp án dạng bảng chuẩn, dùng để chấm điểm | **Có** |
| `qrels_dev.txt` | Đáp án của tập dev | **Có** |
| `steps_*.jsonl` | Các bước con của câu hỏi | Không — thuộc Track 1b |
| `qrels_steps_*.txt` | Đáp án theo từng bước | Không — thuộc Track 1b |
| `guidance_*.jsonl` | Chú thích về yếu tố thời gian | Không bắt buộc |

## 4. Quy mô dữ liệu

In [4]:
print(f"{'domain':<12}{'documents':>12}{'train':>9}{'dev':>7}")
print("-" * 40)

total_docs = 0
total_train = 0
total_dev = 0

for name in domains:
    n_docs = sum(1 for _ in open(DATA / name / "documents.jsonl", encoding="utf-8"))
    n_train = sum(1 for _ in open(DATA / name / "examples_train.jsonl", encoding="utf-8"))
    n_dev = sum(1 for _ in open(DATA / name / "examples_dev.jsonl", encoding="utf-8"))

    total_docs += n_docs
    total_train += n_train
    total_dev += n_dev

    print(f"{name:<12}{n_docs:>12,}{n_train:>9}{n_dev:>7}")

print("-" * 40)
print(f"{'TOTAL':<12}{total_docs:>12,}{total_train:>9}{total_dev:>7}")

domain         documents    train    dev
----------------------------------------
bitcoin          153,291       70     30
cardano           87,201       36     15
economics         93,756       58     25
genealogy        156,228       80     35
history          356,493      561    240
hsm              213,818      105     45
iota              10,372        7      3
law               43,288       24     11
monero            85,093       46     19
politics         183,394      105     45
quant             28,785       24     10
travel           177,677       70     30
workplace         64,659       25     11
----------------------------------------
TOTAL          1,654,055     1211    519


Bảng trên cho thấy số câu hỏi giữa các nhóm **chênh lệch rất lớn**.
Nhóm nhiều nhất gấp nhóm ít nhất khoảng 80 lần. Mục 9 sẽ giải thích vì sao
điều này quan trọng.

Kho văn bản là **chung cho cả hai tập** train và dev. Kho không bị chia nhỏ —
mọi truy hồi đều chạy trên toàn bộ kho của nhóm.

## 5. Cấu trúc của một câu hỏi

In [5]:
with open(DATA / "iota" / "examples_train.jsonl", encoding="utf-8") as f:
    example = json.loads(f.readline())

print("Fields:", list(example.keys()))
print()
print("id             :", example["id"])
print("gold documents :", len(example["gold_ids"]))

Fields: ['id', 'query', 'gold_ids', 'gold_answers']

id             : 2497_799
gold documents : 5


In [6]:
print("QUERY TEXT:")
print("=" * 72)
print(example["query"][:900])

QUERY TEXT:
How long do Zero Value Transactions last?
<p>Let s say we have some critical information stored as Zero Value transactions on the Tangle. How long does it last? </p>

<p>Which robust approaches do you recommend to keep a copy of those transactions?</p>

<p>I am using the testnet.</p>



Câu hỏi **không phải một câu ngắn gọn**, mà là **nguyên một bài đăng
trên diễn đàn Stack Exchange**, gồm:

- một dòng tiêu đề ngắn,
- một phần thân dài giải thích bối cảnh,
- mã HTML dùng để định dạng trang web (`<p>`, `<a href=...>`).

Đặc điểm này ảnh hưởng trực tiếp tới thiết kế hệ thống, vì phần thân dài có thể
làm loãng nội dung chính.

In [7]:
# Split the query into its title and its body.
# The title is everything before the first <p> tag.
match = re.search(r"<p>", example["query"])
title = example["query"][:match.start()] if match else example["query"]
body = example["query"][match.start():] if match else ""

print("TITLE:")
print(" ", title.strip())
print()
print(f"Title length : {len(title):>7,} characters")
print(f"Body length  : {len(body):>7,} characters")
print(f"Ratio        : body is {len(body) / max(len(title), 1):.1f}x the title")

TITLE:
  How long do Zero Value Transactions last?

Title length :      42 characters
Body length  :     243 characters
Ratio        : body is 5.8x the title


## 6. Cấu trúc của một văn bản

In [8]:
with open(DATA / "iota" / "documents.jsonl", encoding="utf-8") as f:
    document = json.loads(f.readline())

print("Fields:", list(document.keys()))
print()
print("id:", document["id"])
print()
print("CONTENT:")
print("=" * 72)
print(document["content"][:700])

Fields: ['id', 'content']

id: iota/470_68d2fff7515d039b_93.txt

CONTENT:
Sharding is a mechanism that is used to partition a blockchain network or other type of computer network or database. Its purpose is to distribute the network's computational and storage workload across a broader set of devices, or nodes, in order to increase the throughput and transaction speed of the entire system. Each node only maintains information related to its specific shard or partition, and since each node is only responsible for processing a fraction of the overall network's transactional load, the network's overall processing capabilities and resilience can be vastly improved. As a result, the increased transaction speeds made possible through sharding have allowed many blockchai


Văn bản trong kho là **văn bản thuần**, không chứa mã HTML.

Như vậy hai phía được viết theo định dạng khác nhau:

| | Có HTML? |
| --- | --- |
| Câu hỏi | Có |
| Văn bản trong kho | Không |

Sự chênh lệch này cần được xử lý ở bước tiền xử lý.

## 7. Cách ghi nhận đáp án

Đáp án được lưu ở hai nơi với nội dung tương đương:

1. Trường `gold_ids` trong `examples_train.jsonl`
2. File `qrels_train.txt` theo định dạng chuẩn TREC

In [9]:
print("First 5 lines of qrels_train.txt:")
print("=" * 64)

with open(DATA / "iota" / "qrels_train.txt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.rstrip())

print()
print("Column layout:")
print("  1. query id")
print("  2. unused (always 0)")
print("  3. document id")
print("  4. relevance: 1 means the document is a correct answer")

First 5 lines of qrels_train.txt:
2497_799	0	iota/a7dc25f7_0169.txt	1
2497_799	0	iota/44afe271_0170.txt	1
2497_799	0	iota/d3459a40_0171.txt	1
2497_799	0	iota/84765dc2_0172.txt	1
2497_799	0	iota/f3c561ea_0173.txt	1

Column layout:
  1. query id
  2. unused (always 0)
  3. document id
  4. relevance: 1 means the document is a correct answer


In [10]:
# Consistency check: the two sources must agree.
gold_from_qrels = set()

with open(DATA / "iota" / "qrels_train.txt", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            qid, _, doc_id, relevance = line.split()
            if qid == example["id"] and int(relevance) > 0:
                gold_from_qrels.add(doc_id)

gold_from_examples = set(example["gold_ids"])

print(f"From examples_train.jsonl : {len(gold_from_examples)} documents")
print(f"From qrels_train.txt      : {len(gold_from_qrels)} documents")
print(f"Identical                 : {gold_from_examples == gold_from_qrels}")

From examples_train.jsonl : 5 documents
From qrels_train.txt      : 5 documents
Identical                 : True


## 8. Cách chấm điểm

Chỉ số đánh giá chính thức là **nDCG@10**
(normalized Discounted Cumulative Gain at rank 10).

Cách hoạt động:

1. Hệ thống nộp một danh sách văn bản đã xếp hạng.
2. Chỉ **10 vị trí đầu tiên** được tính điểm. Từ vị trí 11 trở đi bị bỏ qua.
3. Văn bản đúng nằm ở vị trí càng cao thì đóng góp điểm càng lớn.
   Cụ thể, văn bản ở vị trí thứ `i` đóng góp `1 / log2(i + 2)` điểm.
4. Tổng điểm được chia cho điểm tối đa có thể đạt, nên kết quả nằm trong
   khoảng từ **0 đến 1**.

In [11]:
def ndcg_at_10(ranked_list, gold_set):
    """Compute nDCG@10.

    ranked_list : list of document ids, ordered by predicted relevance
    gold_set    : set of document ids that are correct answers
    """
    # Actual gain: a correct document at position i contributes 1/log2(i+2)
    actual = 0.0
    for i, doc_id in enumerate(ranked_list[:10]):
        if doc_id in gold_set:
            actual += 1 / log2(i + 2)

    # Ideal gain: every one of the top positions is a correct document
    ideal = 0.0
    for i in range(min(len(gold_set), 10)):
        ideal += 1 / log2(i + 2)

    return actual / ideal if ideal > 0 else 0.0

In [12]:
# Illustration: three documents are correct (A, B, C)
gold = {"A", "B", "C"}

scenarios = [
    ("Perfect      - A, B, C in the top 3",        ["A", "B", "C", "x", "y"]),
    ("Good         - A on top, B and C lower",     ["A", "x", "y", "B", "C"]),
    ("Weak         - all 3 found but near bottom", ["x", "y", "z", "w", "v", "u", "t", "A", "B", "C"]),
    ("Poor         - only one, at the bottom",     ["x", "y", "z", "w", "A"]),
    ("Failed       - none correct",                ["x", "y", "z", "w", "v"]),
]

for label, ranking in scenarios:
    print(f"{label:<46} nDCG@10 = {ndcg_at_10(ranking, gold):.3f}")

Perfect      - A, B, C in the top 3            nDCG@10 = 1.000
Good         - A on top, B and C lower         nDCG@10 = 0.853
Weak         - all 3 found but near bottom     nDCG@10 = 0.425
Poor         - only one, at the bottom         nDCG@10 = 0.182
Failed       - none correct                    nDCG@10 = 0.000


Kết quả minh hoạ cho thấy **vị trí quan trọng hơn số lượng**.

Ba trường hợp đầu đều tìm đủ cả 3 văn bản đúng, nhưng điểm chênh nhau đáng kể
chỉ vì thứ hạng khác nhau. Trường hợp "Weak" tìm đúng đủ 3 văn bản nhưng xếp
ở cuối nên gần như mất hết điểm.

## 9. Cách tổng hợp điểm cuối cùng

Điểm cuối được tính theo phương pháp **trung bình theo nhóm** (macro-average):

1. Tính nDCG@10 trung bình cho **từng nhóm chủ đề**.
2. Lấy trung bình cộng của **13 giá trị** đó.

Hệ quả: **13 nhóm có trọng số bằng nhau**, bất kể số lượng câu hỏi chênh lệch
tới đâu.

Nói cách khác, cải thiện kết quả trên một nhóm ít câu hỏi có giá trị **ngang bằng**
với cải thiện trên một nhóm nhiều câu hỏi.

In [13]:
query_counts = {}
for name in domains:
    query_counts[name] = sum(
        1 for _ in open(DATA / name / "examples_train.jsonl", encoding="utf-8")
    )

largest = max(query_counts, key=query_counts.get)
smallest = min(query_counts, key=query_counts.get)
total = sum(query_counts.values())

print(f"Largest domain  : {largest:<12}{query_counts[largest]:>5} queries"
      f"  ({100 * query_counts[largest] / total:>5.1f}% of all queries)")
print(f"Smallest domain : {smallest:<12}{query_counts[smallest]:>5} queries"
      f"  ({100 * query_counts[smallest] / total:>5.1f}% of all queries)")
print()
print(f"Ratio: {query_counts[largest] / query_counts[smallest]:.0f}x")
print()
print(f"Yet under macro-averaging each domain carries "
      f"{100 / len(domains):.1f}% of the final score.")

Largest domain  : history       561 queries  ( 46.3% of all queries)
Smallest domain : iota            7 queries  (  0.6% of all queries)

Ratio: 80x

Yet under macro-averaging each domain carries 7.7% of the final score.


## 10. Mốc điểm tham chiếu

Ban tổ chức đã công bố kết quả của một phương pháp cơ sở (baseline) mang tên
**BM25**, để làm mốc so sánh.

BM25 là phương pháp truy hồi cổ điển dựa trên thống kê từ vựng: đếm số từ chung
giữa câu hỏi và văn bản, trong đó từ hiếm được tính trọng số cao hơn. Phương pháp
này **không hiểu ngữ nghĩa** và **không xử lý được yếu tố thời gian**.

| Tập dữ liệu | nDCG@10 của BM25 |
| --- | --- |
| `train` (1.211 câu hỏi) | **0,0879** |
| `dev` (519 câu hỏi) | 0,0967 |

Vì quá trình phát triển diễn ra trên tập `train`, **mốc cần vượt là 0,0879**.

Bài báo gốc của bộ dữ liệu (TEMPO) công bố con số 0,108, nhưng đó là kết quả
trên toàn bộ tập train và dev gộp lại, nên không so sánh trực tiếp được.
Các hệ thống tốt nhất trong bài báo đó đạt khoảng **0,30**.

| Mức | nDCG@10 |
| --- | --- |
| BM25 cơ sở | 0,088 |
| Mục tiêu nhóm dẫn đầu | ~0,30 |

*Nguồn: `RETECO/starter_kit/BASELINE_RESULTS.md`*

In [14]:
# Official BM25 baseline per domain, on the train split.
# Source: RETECO/starter_kit/BASELINE_RESULTS.md
baseline_train = {
    "bitcoin": 0.0695, "cardano": 0.1349, "economics": 0.0382,
    "genealogy": 0.1003, "history": 0.0691, "hsm": 0.1627,
    "iota": 0.0199, "law": 0.0943, "monero": 0.0278,
    "politics": 0.2792, "quant": 0.0255, "travel": 0.0429,
    "workplace": 0.0777,
}

print("BM25 baseline on train, sorted from highest to lowest:\n")
for name, score in sorted(baseline_train.items(), key=lambda kv: -kv[1]):
    bar = "#" * int(score * 100)
    print(f"  {name:<12}{score:.4f}  {bar}")

macro_average = sum(baseline_train.values()) / len(baseline_train)
print(f"\nMacro-average over {len(baseline_train)} domains: {macro_average:.4f}")
print(f"Hardest domain is {min(baseline_train, key=baseline_train.get)}, "
      f"easiest is {max(baseline_train, key=baseline_train.get)} "
      f"({max(baseline_train.values()) / min(baseline_train.values()):.0f}x apart)")

BM25 baseline on train, sorted from highest to lowest:

  politics    0.2792  ###########################
  hsm         0.1627  ################
  cardano     0.1349  #############
  genealogy   0.1003  ##########
  law         0.0943  #########
  workplace   0.0777  #######
  bitcoin     0.0695  ######
  history     0.0691  ######
  travel      0.0429  ####
  economics   0.0382  ###
  monero      0.0278  ##
  quant       0.0255  ##
  iota        0.0199  #

Macro-average over 13 domains: 0.0878
Hardest domain is iota, easiest is politics (14x apart)


Mức độ khó giữa các nhóm chênh nhau khoảng 14 lần. Điều này có hai hệ quả
cho các bước sau:

1. Không thể đánh giá một cải tiến dựa trên kết quả của một nhóm đơn lẻ.
2. Một cải tiến chỉ có tác dụng trên nhóm dễ sẽ không phản ánh chất lượng thực sự.

## 11. Quy định của cuộc thi

| Quy định | Diễn giải |
| --- | --- |
| Phát triển trên tập `train` | Mọi thử nghiệm thực hiện trên các file `*_train` |
| Tập `dev` chỉ dùng **một lần** | Chạy ở bước cuối cùng, sau đó không điều chỉnh hệ thống nữa |
| Chỉ truy hồi trong kho đã cho | Không bổ sung dữ liệu từ nguồn bên ngoài |
| Công khai mô hình đã sử dụng | Khai báo đầy đủ mô hình, API và phiên bản trong báo cáo |

## Tổng kết

| Hạng mục | Nội dung |
| --- | --- |
| Đầu vào | Một bài đăng diễn đàn (tiêu đề + phần thân, có HTML) |
| Đầu ra | Danh sách văn bản được xếp hạng |
| Phạm vi tìm kiếm | Kho văn bản của cùng nhóm chủ đề |
| Số nhóm | 13 |
| Quy mô kho | 1.654.055 văn bản |
| Số câu hỏi | 1.211 (train) + 519 (dev) |
| Chỉ số đánh giá | nDCG@10, trung bình theo nhóm |
| Mốc cơ sở | BM25 = 0,0879 trên `train` |
| Mục tiêu | Khoảng 0,30 |
| Thách thức chính | Ràng buộc thời gian trong câu hỏi |

**Bước tiếp theo:** Notebook 01a + 01b — phân tích khám phá dữ liệu (EDA).